# Chapter 1: The Python Data Model
**From:** *Fluent Python* by Luciano Ramalho

---

## TL;DR

- Python's **data model** (a.k.a. the "metaobject protocol") lets you define **special (dunder) methods** so your objects integrate with built-in syntax: `len()`, `[]`, `for`, `+`, `in`, `print()`, and more.
- Implementing just `__getitem__` and `__len__` on the **FrenchDeck** class gives you indexing, slicing, iteration, membership testing, `random.choice`, `reversed()`, and `sorted()` -- for free.
- Implementing `__add__`, `__mul__`, `__abs__`, `__repr__`, and `__bool__` on a **Vector** class makes it behave like a first-class numeric type with operator overloading.
- Choose `__repr__` over `__str__` when implementing only one string method -- the default `__str__` falls back to `__repr__`, but not vice versa.
- The `collections.abc` module formalizes collection interfaces through ABCs: **Iterable**, **Sized**, **Container** -> **Collection** -> **Sequence**, **Mapping**, **Set**.

---

## 1. The Sequence Protocol: FrenchDeck

A deck of cards built with `collections.namedtuple` and just two special methods: `__len__` and `__getitem__`. This is enough to make it behave like a full Python sequence.

In [1]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit)
                       for suit in self.suits
                       for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

deck = FrenchDeck()
print(f"Deck length: {len(deck)}")
print(f"First card:  {deck[0]}")
print(f"Last card:   {deck[-1]}")

Deck length: 52
First card:  Card(rank='2', suit='spades')
Last card:   Card(rank='A', suit='hearts')


### Slicing

Because `__getitem__` delegates to a list, standard slice syntax works automatically:

In [ ]:
# Top 3 cards
print("Top 3:", deck[:3])

# All four Aces (index 12, then every 13th card)
print("Aces: ", deck[12::13])

### Iteration and `random.choice`

`__getitem__` with integer indices is enough for Python's `for` loop (the legacy sequence protocol). Standard library functions like `random.choice` also work because the deck is a sequence:

In [ ]:
from random import choice, seed

seed(42)  # reproducible
print("Random card:", choice(deck))
print("Random card:", choice(deck))
print("Random card:", choice(deck))

In [ ]:
# Iteration works via __getitem__ fallback
print("First 5 cards via iteration:")
for i, card in enumerate(deck):
    if i >= 5:
        break
    print(f"  {card}")

# Membership testing (in) uses sequential scan
queen_in = Card('Q', 'hearts') in deck
beast_in = Card('7', 'beasts') in deck
print(f"\nQueen of hearts in deck? {queen_in}")
print(f"7 of beasts in deck?    {beast_in}")

### Sorting with a custom key

A ranking function maps each card to an integer for sorting:

In [ ]:
suit_values = dict(spades=3, hearts=2, diamonds=1, clubs=0)

def spades_high(card):
    rank_value = FrenchDeck.ranks.index(card.rank)
    return rank_value * len(suit_values) + suit_values[card.suit]

# Show the 3 lowest and 3 highest cards
sorted_deck = sorted(deck, key=spades_high)
print("Lowest 3: ", sorted_deck[:3])
print("Highest 3:", sorted_deck[-3:])

---

## 2. Emulating Numeric Types: Vector

A 2D vector class that supports `+`, `*`, `abs()`, `repr()`, and `bool()` via special methods:

In [ ]:
import math

class Vector:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Vector({self.x!r}, {self.y!r})'

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __bool__(self):
        return bool(abs(self))

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)

# Demonstrate each operation
v1 = Vector(2, 4)
v2 = Vector(2, 1)
print(f"v1 = {v1!r}")
print(f"v2 = {v2!r}")
print(f"v1 + v2 = {v1 + v2!r}")

In [ ]:
v = Vector(3, 4)
print(f"abs(Vector(3, 4)) = {abs(v)}")   # magnitude: 5.0
print(f"v * 3 = {v * 3!r}")              # scalar multiplication
print(f"abs(v * 3) = {abs(v * 3)}")      # 15.0

---

## 3. String Representation: `__repr__` vs `__str__`

- **`__repr__`**: unambiguous, developer-facing (console, debugger, `!r`). Should look like a valid constructor call if possible.
- **`__str__`**: readable, user-facing (`print()`, `str()`).
- If you implement only one, choose **`__repr__`** -- `object.__str__` falls back to `__repr__`, but not the reverse.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point({self.x!r}, {self.y!r})'

    def __str__(self):
        return f'({self.x}, {self.y})'

p = Point(3, 4)
print(f"repr(p) = {repr(p)}")   # developer view
print(f"str(p)  = {str(p)}")    # user view
print("print calls __str__:")
print(p)

# In f-strings: !r forces repr, default uses str
print(f"f-string default: {p}")
print(f"f-string !r:      {p!r}")

In [ ]:
# What happens with ONLY __repr__ (no __str__)?
class Widget:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f'Widget({self.name!r})'

w = Widget("gear")
print(f"repr: {repr(w)}")
print(f"str:  {str(w)}")   # falls back to __repr__
print("print also uses __repr__ fallback:")
print(w)

---

## 4. Boolean Value of Custom Types

Python's truth-value resolution order:

1. **`__bool__()`** -- if defined, must return `True` or `False`
2. **`__len__()`** fallback -- returns `False` if 0, `True` otherwise
3. **Default** -- object is always truthy

In [ ]:
# The Vector's __bool__ uses magnitude
v_zero = Vector(0, 0)
v_nonzero = Vector(1, 0)
print(f"bool(Vector(0, 0)) = {bool(v_zero)}")     # False
print(f"bool(Vector(1, 0)) = {bool(v_nonzero)}")  # True

In [ ]:
# Demonstrate the __len__ fallback
class Bag:
    def __init__(self, items):
        self._items = list(items)
    def __len__(self):
        return len(self._items)

empty_bag = Bag([])
full_bag = Bag([1, 2, 3])
print(f"bool(empty_bag) = {bool(empty_bag)}")   # False (__len__ returns 0)
print(f"bool(full_bag)  = {bool(full_bag)}")    # True  (__len__ returns 3)

# Default: no __bool__, no __len__ -> always truthy
class Placeholder:
    pass

print(f"bool(Placeholder()) = {bool(Placeholder())}")  # True

---

## 5. Collection API and ABCs

The `collections.abc` module defines abstract base classes that formalize collection interfaces:

| ABC | Required method | Provides |
|-----|----------------|----------|
| `Iterable` | `__iter__` | `for` loops, unpacking |
| `Sized` | `__len__` | `len()` |
| `Container` | `__contains__` | `in` operator |
| **`Collection`** | all three above | unified base (Python 3.6+) |

Specialized ABCs extend `Collection`:
- **`Sequence`** (`list`, `str`, `tuple`) -- adds `__getitem__`, `__reversed__`, `index`, `count`
- **`Mapping`** (`dict`) -- adds `keys`, `values`, `items`, `get`
- **`Set`** (`set`, `frozenset`) -- adds `__and__`, `__or__`, `__xor__`, `isdisjoint`

In [ ]:
from collections.abc import Sized, Iterable, Container, Sequence

# Structural subtyping: no inheritance needed for Sized
class MyStack:
    def __init__(self):
        self._data = []
    def push(self, item):
        self._data.append(item)
    def __len__(self):
        return len(self._data)

s = MyStack()
print(f"isinstance(MyStack(), Sized) = {isinstance(s, Sized)}")  # True!

# But FrenchDeck does NOT pass the Sequence check (no __subclasshook__)
print(f"isinstance(FrenchDeck(), Sequence) = {isinstance(FrenchDeck(), Sequence)}")

In [ ]:
# Inheriting from Sequence gives you concrete methods for free
from collections.abc import Sequence as SeqABC

class LetterSeq(SeqABC):
    def __init__(self, text):
        self._data = list(text)

    def __getitem__(self, index):
        return self._data[index]

    def __len__(self):
        return len(self._data)

letters = LetterSeq("python")
print(f"Length:      {len(letters)}")
print(f"letters[0]: {letters[0]!r}")
p_in = 'p' in letters
print(f"'p' in letters: {p_in}")       # inherited __contains__
h_idx = letters.index('h')
print(f"index('h'):     {h_idx}")       # inherited index
t_cnt = letters.count('t')
print(f"count('t'):     {t_cnt}")        # inherited count

---

## Try It Yourself

### Exercise 1: Temperature class

Create a `Temperature` class that:
- Stores degrees in Celsius
- `__repr__` returns e.g. `Temperature(100)`
- `__str__` returns e.g. `100°C`
- `__bool__` returns `False` for absolute zero (-273.15) or below, `True` otherwise
- `__add__` adds two temperatures (returns new Temperature)

Test it with: `print()`, `repr()`, `bool()`, `if`, and `+`.

In [ ]:
# Exercise 1: Your solution here
class Temperature:
    pass  # TODO: implement __init__, __repr__, __str__, __bool__, __add__

# Uncomment to test:
# t1 = Temperature(100)
# t2 = Temperature(50)
# print(repr(t1))            # Temperature(100)
# print(t1)                  # 100°C
# print(t1 + t2)             # 150°C
# print(bool(t1))            # True
# print(bool(Temperature(-300)))  # False

<details>
<summary>Click to reveal solution</summary>

```python
class Temperature:
    def __init__(self, celsius=0):
        self.celsius = celsius

    def __repr__(self):
        return f'Temperature({self.celsius!r})'

    def __str__(self):
        return f'{self.celsius}°C'

    def __bool__(self):
        return self.celsius > -273.15

    def __add__(self, other):
        return Temperature(self.celsius + other.celsius)
```
</details>

### Exercise 2: Playlist sequence

Create a `Playlist` class that:
- Takes a list of song names
- Implements `__len__` and `__getitem__` (the sequence protocol)
- Verify that `len()`, indexing, slicing, iteration, `in`, `reversed()`, and `random.choice()` all work

In [ ]:
# Exercise 2: Your solution here
class Playlist:
    pass  # TODO: implement __init__, __len__, __getitem__

# Uncomment to test:
# songs = Playlist(["Bohemian Rhapsody", "Stairway to Heaven", "Hotel California",
#                   "Imagine", "Smells Like Teen Spirit"])
# print(f"Length: {len(songs)}")
# print(f"First:  {songs[0]}")
# print(f"Last 2: {songs[-2:]}")
# print(f"'Imagine' in playlist: {'Imagine' in songs}")
# for s in reversed(songs):
#     print(f"  {s}")

<details>
<summary>Click to reveal solution</summary>

```python
class Playlist:
    def __init__(self, songs):
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, position):
        return self._songs[position]
```
</details>

### Exercise 3: Money with operators

Create a `Money` class that:
- Stores `amount` (float) and `currency` (str, e.g. "USD")
- `__repr__` returns e.g. `Money(9.99, 'USD')`
- `__add__` adds two Money objects (raise `ValueError` if currencies differ)
- `__mul__` multiplies by a scalar
- `__bool__` returns `False` if amount is 0

In [ ]:
# Exercise 3: Your solution here
class Money:
    pass  # TODO: implement

# Uncomment to test:
# a = Money(9.99, "USD")
# b = Money(5.01, "USD")
# print(repr(a))           # Money(9.99, 'USD')
# print(repr(a + b))       # Money(15.0, 'USD')
# print(repr(a * 3))       # Money(29.97, 'USD')
# print(bool(Money(0, "USD")))  # False
# print(bool(a))                # True

<details>
<summary>Click to reveal solution</summary>

```python
class Money:
    def __init__(self, amount, currency):
        self.amount = amount
        self.currency = currency

    def __repr__(self):
        return f'Money({self.amount!r}, {self.currency!r})'

    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError(f"Cannot add {self.currency} and {other.currency}")
        return Money(round(self.amount + other.amount, 2), self.currency)

    def __mul__(self, scalar):
        return Money(round(self.amount * scalar, 2), self.currency)

    def __bool__(self):
        return self.amount != 0
```
</details>

---

## Key Takeaways

1. **The Python Data Model is an API contract.** Implement special methods, and the interpreter calls them at the right time -- you almost never call them directly.

2. **Two methods unlock sequence behavior.** `__getitem__` + `__len__` give you indexing, slicing, iteration, `in`, `reversed()`, `sorted()`, and `random.choice()` -- the FrenchDeck pattern.

3. **Operator overloading creates natural APIs.** `__add__`, `__mul__`, `__abs__` let your objects respond to `+`, `*`, `abs()` just like built-in numeric types.

4. **`__repr__` over `__str__`.** If you implement only one, choose `__repr__`. It serves the debugger, console, and logging. `__str__` falls back to `__repr__`, but not vice versa.

5. **Boolean resolution: `__bool__` -> `__len__` -> always truthy.** Collections get truthiness from `__len__` automatically. Non-collections should implement `__bool__` explicitly.

6. **Composition over inheritance.** FrenchDeck delegates to an internal list without inheriting from `list`. This pattern keeps the interface clean and focused.

7. **ABCs formalize protocols.** `collections.abc` defines `Iterable`, `Sized`, `Container`, `Collection`, `Sequence`, `Mapping`, and `Set` -- but duck typing means you rarely need to inherit from them.

---

## See Also

Detailed wiki articles for each concept in this chapter:

- [[special-dunder-methods]] -- Overview of the Python Data Model and all categories of special methods
- [[sequence-protocol-french-deck]] -- Deep dive into the FrenchDeck example and the sequence protocol
- [[numeric-type-emulation]] -- The Vector class: operator overloading for custom numeric types
- [[string-representation]] -- `__repr__` vs `__str__` vs `__format__` in detail
- [[boolean-value-custom-types]] -- Truth-value testing: `__bool__`, `__len__` fallback, and built-in falsy values
- [[collection-api-abcs]] -- The `collections.abc` hierarchy: Iterable, Sized, Container, Collection, Sequence, Mapping, Set